# IoT Simulated Pipeline Analytics Exploration

This notebook provides comprehensive analysis and visualization of IoT sensor data from the simulated pipeline. We'll explore sensor readings, analyze device health, detect anomalies, and build predictive models for maintenance scheduling.

## Dataset Overview
- **Source**: PostgreSQL database (iot_analytics)
- **Data**: Simulated IoT sensor readings (temperature, humidity, pressure)
- **Devices**: 100+ simulated IoT devices
- **Features**: Device metadata, sensor readings, alerts, ML predictions
- **Time Range**: Real-time and historical data

## Analysis Goals
1. Explore sensor data patterns and trends
2. Analyze device health and performance
3. Visualize anomalies and alerts
4. Build predictive maintenance models
5. Create interactive dashboards

## 1. Import Required Libraries
Import essential libraries for data analysis, visualization, and machine learning.

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

# Database connectivity
import psycopg2
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Statistical analysis
from scipy import stats
from scipy.stats import zscore

# Set plotting styles
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")

## 2. Load and Inspect Data
Connect to the PostgreSQL database and load IoT sensor data for analysis.

In [ ]:
# Load environment variables
load_dotenv()

# Database connection parameters
DB_HOST = os.getenv('POSTGRES_HOST', 'localhost')
DB_PORT = os.getenv('POSTGRES_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'iot_analytics')
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')

# Create database connection
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

print(f"🔌 Connecting to database: {DB_HOST}:{DB_PORT}/{DB_NAME}")

try:
    # Test connection
    test_query = "SELECT COUNT(*) as total_records FROM sensor_readings"
    test_result = pd.read_sql(test_query, engine)
    print(f"✅ Database connection successful!")
    print(f"📊 Total sensor readings: {test_result['total_records'].iloc[0]:,}")
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    raise

In [ ]:
# Load IoT sensor data (last 7 days for performance)
sensor_query = """
SELECT 
    sr.*,
    dm.device_type,
    dm.location,
    dm.status,
    EXTRACT(HOUR FROM sr.timestamp) as hour_of_day,
    EXTRACT(DOW FROM sr.timestamp) as day_of_week
FROM sensor_readings sr
JOIN device_metadata dm ON sr.device_id = dm.device_id
WHERE sr.timestamp >= NOW() - INTERVAL '7 days'
ORDER BY sr.timestamp DESC
LIMIT 100000
"""

print("📊 Loading sensor data...")
sensor_data = pd.read_sql(sensor_query, engine)

# Load device metadata
device_query = "SELECT * FROM device_metadata"
device_metadata = pd.read_sql(device_query, engine)

# Load anomaly detections
anomaly_query = """
SELECT * FROM anomaly_detections 
WHERE detected_at >= NOW() - INTERVAL '7 days'
ORDER BY detected_at DESC
"""
anomaly_data = pd.read_sql(anomaly_query, engine)

# Load ML predictions
ml_query = """
SELECT * FROM ml_predictions 
WHERE timestamp >= NOW() - INTERVAL '7 days'
ORDER BY timestamp DESC
"""
ml_predictions = pd.read_sql(ml_query, engine)

print(f"✅ Data loaded successfully!")
print(f"📊 Sensor readings: {len(sensor_data):,} records")
print(f"🏭 Devices: {len(device_metadata):,} devices")
print(f"🚨 Anomalies: {len(anomaly_data):,} detected")
print(f"🤖 ML predictions: {len(ml_predictions):,} predictions")

# Display basic info
print(f"\n📅 Data time range: {sensor_data['timestamp'].min()} to {sensor_data['timestamp'].max()}")
print(f"🏷️  Device types: {', '.join(device_metadata['device_type'].unique())}")
print(f"📍 Locations: {len(device_metadata['location'].unique())} unique locations")

In [ ]:
# Inspect sensor data structure
print("🔍 Sensor Data Structure:")
print(sensor_data.info())
print(f"\n📊 Shape: {sensor_data.shape[0]:,} rows × {sensor_data.shape[1]} columns")

# Display first few rows
print("\n👀 First 5 sensor readings:")
sensor_data.head()

## 3. Data Preprocessing
Clean and prepare the data for analysis by handling missing values, data types, and outliers.

In [ ]:
# Convert timestamp to datetime
sensor_data['timestamp'] = pd.to_datetime(sensor_data['timestamp'])
sensor_data = sensor_data.sort_values('timestamp')

# Check for missing values
print("🔍 Missing Values Analysis:")
missing_values = sensor_data.isnull().sum()
print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("✅ No missing values found!")
else:
    print(f"⚠️  Found {missing_values.sum()} missing values")

# Basic statistics for sensor readings
print("\n📊 Sensor Reading Statistics:")
numeric_columns = ['temperature', 'humidity', 'pressure', 'battery_level', 'signal_strength']
sensor_stats = sensor_data[numeric_columns].describe()
print(sensor_stats)

# Check for outliers using IQR method
print("\n🎯 Outlier Detection:")
for col in numeric_columns:
    Q1 = sensor_data[col].quantile(0.25)
    Q3 = sensor_data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = sensor_data[(sensor_data[col] < lower_bound) | (sensor_data[col] > upper_bound)]
    outlier_percentage = (len(outliers) / len(sensor_data)) * 100
    
    print(f"  {col}: {len(outliers):,} outliers ({outlier_percentage:.2f}%)")

# Data quality metrics
print(f"\n✅ Data Quality Summary:")
print(f"  • Total records: {len(sensor_data):,}")
print(f"  • Unique devices: {sensor_data['device_id'].nunique()}")
print(f"  • Date range: {sensor_data['timestamp'].dt.date.min()} to {sensor_data['timestamp'].dt.date.max()}")
print(f"  • Completeness: {((sensor_data.notna().sum().sum()) / (len(sensor_data) * len(sensor_data.columns)) * 100):.2f}%")

## 4. Feature Engineering
Create new features and prepare data for machine learning models.

In [ ]:
# Create additional time-based features
sensor_data['date'] = sensor_data['timestamp'].dt.date
sensor_data['hour'] = sensor_data['timestamp'].dt.hour
sensor_data['day_name'] = sensor_data['timestamp'].dt.day_name()
sensor_data['is_weekend'] = sensor_data['day_of_week'].isin([5, 6]).astype(int)
sensor_data['is_night'] = ((sensor_data['hour'] < 6) | (sensor_data['hour'] > 22)).astype(int)

# Create sensor interaction features
sensor_data['temp_humidity_ratio'] = sensor_data['temperature'] / (sensor_data['humidity'] + 1e-6)
sensor_data['temp_pressure_interaction'] = sensor_data['temperature'] * sensor_data['pressure'] / 1000
sensor_data['battery_signal_score'] = sensor_data['battery_level'] * sensor_data['signal_strength'] / 100

# Create device health indicators
sensor_data['low_battery'] = (sensor_data['battery_level'] < 20).astype(int)
sensor_data['poor_signal'] = (sensor_data['signal_strength'] < 30).astype(int)
sensor_data['extreme_temp'] = ((sensor_data['temperature'] < 0) | (sensor_data['temperature'] > 50)).astype(int)

# Calculate rolling averages (per device)
sensor_data = sensor_data.sort_values(['device_id', 'timestamp'])
window_size = 6  # 6-hour rolling window

for col in ['temperature', 'humidity', 'pressure']:
    sensor_data[f'{col}_rolling_mean'] = sensor_data.groupby('device_id')[col].rolling(
        window=window_size, min_periods=1).mean().reset_index(0, drop=True)
    sensor_data[f'{col}_rolling_std'] = sensor_data.groupby('device_id')[col].rolling(
        window=window_size, min_periods=1).std().reset_index(0, drop=True)

# Calculate z-scores for anomaly detection
sensor_data['temp_zscore'] = sensor_data.groupby('device_id')['temperature'].transform(lambda x: zscore(x, nan_policy='omit'))
sensor_data['humidity_zscore'] = sensor_data.groupby('device_id')['humidity'].transform(lambda x: zscore(x, nan_policy='omit'))
sensor_data['pressure_zscore'] = sensor_data.groupby('device_id')['pressure'].transform(lambda x: zscore(x, nan_policy='omit'))

print("🔧 Feature Engineering Complete!")
print(f"📊 New dataset shape: {sensor_data.shape}")
print(f"🆕 Added features: {sensor_data.shape[1] - len(sensor_stats.columns) - 10} new columns")

# Display new feature statistics
new_features = ['temp_humidity_ratio', 'temp_pressure_interaction', 'battery_signal_score', 
                'temperature_rolling_mean', 'humidity_rolling_mean', 'pressure_rolling_mean']
print(f"\n📈 New Feature Statistics:")
sensor_data[new_features].describe()

## 5. Model Training
Build machine learning models for predictive maintenance and anomaly detection.

In [ ]:
# Prepare data for predictive maintenance model
# Target: predict battery life remaining (simplified model)
sensor_data['battery_life_remaining'] = sensor_data['battery_level'] / 2.5  # Simplified: days remaining

# Select features for the model
feature_columns = [
    'temperature', 'humidity', 'pressure', 'signal_strength', 
    'hour', 'day_of_week', 'is_weekend', 'is_night',
    'temp_humidity_ratio', 'temp_pressure_interaction', 
    'temperature_rolling_mean', 'humidity_rolling_mean', 'pressure_rolling_mean'
]

# Prepare training data (remove rows with NaN values)
training_data = sensor_data.dropna(subset=feature_columns + ['battery_life_remaining'])

X = training_data[feature_columns]
y = training_data['battery_life_remaining']

print(f"🎯 Training data prepared:")
print(f"  • Features: {len(feature_columns)}")
print(f"  • Samples: {len(X):,}")
print(f"  • Target range: {y.min():.2f} to {y.max():.2f} days")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n📊 Data split:")
print(f"  • Training set: {len(X_train):,} samples")
print(f"  • Test set: {len(X_test):,} samples")

# Train Random Forest model for battery life prediction
print(f"\n🤖 Training Random Forest model...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

print("✅ Model training complete!")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🎯 Top 5 Most Important Features:")
for idx, row in feature_importance.head().iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

## 6. Model Evaluation
Evaluate the trained model using appropriate metrics and visualizations.

In [ ]:
# Make predictions on test set
y_pred = rf_model.predict(X_test_scaled)

# Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test - y_pred))
r2 = rf_model.score(X_test_scaled, y_test)

print("📊 Model Performance Metrics:")
print(f"  • Root Mean Square Error (RMSE): {rmse:.4f} days")
print(f"  • Mean Absolute Error (MAE): {mae:.4f} days")
print(f"  • R² Score: {r2:.4f}")

# Create evaluation visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Actual vs Predicted scatter plot
axes[0, 0].scatter(y_test, y_pred, alpha=0.6, color='blue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Battery Life (days)')
axes[0, 0].set_ylabel('Predicted Battery Life (days)')
axes[0, 0].set_title('Actual vs Predicted Values')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals plot
residuals = y_test - y_pred
axes[0, 1].scatter(y_pred, residuals, alpha=0.6, color='green')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted Values')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals Plot')
axes[0, 1].grid(True, alpha=0.3)

# 3. Feature importance plot
top_features = feature_importance.head(10)
axes[1, 0].barh(top_features['feature'], top_features['importance'], color='orange')
axes[1, 0].set_xlabel('Feature Importance')
axes[1, 0].set_title('Top 10 Feature Importance')
axes[1, 0].grid(True, alpha=0.3)

# 4. Prediction distribution
axes[1, 1].hist(y_test, bins=30, alpha=0.7, label='Actual', color='blue')
axes[1, 1].hist(y_pred, bins=30, alpha=0.7, label='Predicted', color='red')
axes[1, 1].set_xlabel('Battery Life (days)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Actual vs Predicted Values')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Model performance summary
print(f"\n✅ Model Evaluation Summary:")
print(f"  • The model explains {r2*100:.1f}% of the variance in battery life")
print(f"  • Average prediction error: ±{mae:.2f} days")
print(f"  • Model is {'good' if r2 > 0.7 else 'moderate' if r2 > 0.5 else 'needs improvement'} (R² = {r2:.3f})")

## 7. Make Predictions
Use the trained model to make predictions on new data and identify devices requiring maintenance.

In [ ]:
# Generate predictions for all devices
latest_data = sensor_data.dropna(subset=feature_columns).groupby('device_id').tail(1)
X_latest = latest_data[feature_columns]
X_latest_scaled = scaler.transform(X_latest)

# Make predictions
predictions = rf_model.predict(X_latest_scaled)

# Create maintenance schedule
maintenance_schedule = pd.DataFrame({
    'device_id': latest_data['device_id'].values,
    'device_type': latest_data['device_type'].values,
    'location': latest_data['location'].values,
    'current_battery': latest_data['battery_level'].values,
    'predicted_days_remaining': predictions,
    'maintenance_priority': pd.cut(predictions, 
                                 bins=[0, 5, 15, 30, float('inf')], 
                                 labels=['Critical', 'High', 'Medium', 'Low'])
})

# Sort by priority
maintenance_schedule = maintenance_schedule.sort_values('predicted_days_remaining')

print("🔧 Predictive Maintenance Schedule Generated!")
print(f"📊 Total devices analyzed: {len(maintenance_schedule)}")

# Show priority distribution
priority_counts = maintenance_schedule['maintenance_priority'].value_counts()
print(f"\n🎯 Maintenance Priority Distribution:")
for priority, count in priority_counts.items():
    print(f"  • {priority}: {count} devices")

# Display critical devices (need immediate attention)
critical_devices = maintenance_schedule[maintenance_schedule['maintenance_priority'] == 'Critical']
print(f"\n🚨 Critical Devices (Immediate Maintenance Required):")
if len(critical_devices) > 0:
    print(critical_devices[['device_id', 'device_type', 'location', 'current_battery', 'predicted_days_remaining']].to_string(index=False))
else:
    print("  ✅ No critical devices found!")

# Display top 10 devices needing maintenance
print(f"\n⚠️  Top 10 Devices Needing Maintenance:")
top_maintenance = maintenance_schedule.head(10)
print(top_maintenance[['device_id', 'device_type', 'location', 'current_battery', 'predicted_days_remaining', 'maintenance_priority']].to_string(index=False))

# Create interactive maintenance dashboard
fig = px.scatter(
    maintenance_schedule,
    x='current_battery',
    y='predicted_days_remaining',
    color='maintenance_priority',
    size='current_battery',
    hover_data=['device_id', 'device_type', 'location'],
    title='Predictive Maintenance Dashboard',
    labels={
        'current_battery': 'Current Battery Level (%)',
        'predicted_days_remaining': 'Predicted Days Until Maintenance',
        'maintenance_priority': 'Priority'
    },
    color_discrete_map={
        'Critical': 'red',
        'High': 'orange', 
        'Medium': 'yellow',
        'Low': 'green'
    }
)

fig.update_layout(
    width=800,
    height=600,
    showlegend=True
)

fig.show()

print(f"\n✅ Predictive maintenance analysis complete!")
print(f"📈 Model accuracy: {r2*100:.1f}%")
print(f"🎯 Use this schedule to proactively maintain devices before failures occur")

## 8. Advanced Analytics & Visualizations
Additional insights including time series analysis, anomaly detection, and interactive dashboards.

In [ ]:
# Time Series Analysis
print("📈 Creating Time Series Visualizations...")

# Sample devices for detailed analysis
sample_devices = sensor_data['device_id'].unique()[:5]
sample_data = sensor_data[sensor_data['device_id'].isin(sample_devices)]

# Create interactive time series plot
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Temperature Trends', 'Humidity Trends', 'Battery Levels'),
    vertical_spacing=0.08
)

colors = px.colors.qualitative.Set1

for i, device in enumerate(sample_devices):
    device_data = sample_data[sample_data['device_id'] == device].sort_values('timestamp')
    color = colors[i % len(colors)]
    
    fig.add_trace(
        go.Scatter(x=device_data['timestamp'], y=device_data['temperature'],
                  name=f'{device} - Temp', line=dict(color=color)),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=device_data['timestamp'], y=device_data['humidity'],
                  name=f'{device} - Humidity', line=dict(color=color, dash='dash')),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=device_data['timestamp'], y=device_data['battery_level'],
                  name=f'{device} - Battery', line=dict(color=color, dash='dot')),
        row=3, col=1
    )

fig.update_layout(height=800, title_text="IoT Device Time Series Analysis")
fig.update_xaxes(title_text="Time", row=3, col=1)
fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
fig.update_yaxes(title_text="Humidity (%)", row=2, col=1)
fig.update_yaxes(title_text="Battery Level (%)", row=3, col=1)
fig.show()

# Device Health Heatmap
print("\n🔥 Creating Device Health Heatmap...")

# Calculate device health metrics
device_health = sensor_data.groupby('device_id').agg({
    'temperature': ['mean', 'std'],
    'humidity': ['mean', 'std'],
    'pressure': ['mean', 'std'],
    'battery_level': ['mean', 'min'],
    'signal_strength': ['mean', 'min']
}).round(2)

# Flatten column names
device_health.columns = ['_'.join(col).strip() for col in device_health.columns]

# Create heatmap for first 20 devices
health_sample = device_health.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(health_sample.T, annot=True, cmap='RdYlGn_r', center=0, ax=ax)
ax.set_title('Device Health Metrics Heatmap')
ax.set_xlabel('Device ID')
ax.set_ylabel('Health Metrics')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Anomaly Detection Visualization
print("\n🚨 Anomaly Detection Analysis...")

if len(anomaly_data) > 0:
    # Anomaly timeline
    anomaly_data['detected_at'] = pd.to_datetime(anomaly_data['detected_at'])
    anomaly_timeline = anomaly_data.groupby([
        anomaly_data['detected_at'].dt.date, 'anomaly_type'
    ]).size().reset_index(name='count')
    
    fig = px.bar(
        anomaly_timeline, 
        x='detected_at', 
        y='count',
        color='anomaly_type',
        title='Anomaly Detection Timeline',
        labels={'detected_at': 'Date', 'count': 'Number of Anomalies'}
    )
    fig.show()
    
    # Anomaly distribution by device type
    if not anomaly_data.empty and 'device_id' in anomaly_data.columns:
        anomaly_with_metadata = anomaly_data.merge(
            device_metadata[['device_id', 'device_type', 'location']], 
            on='device_id', 
            how='left'
        )
        
        anomaly_by_type = anomaly_with_metadata.groupby(['device_type', 'anomaly_type']).size().reset_index(name='count')
        
        fig = px.sunburst(
            anomaly_by_type,
            path=['device_type', 'anomaly_type'],
            values='count',
            title='Anomaly Distribution by Device Type'
        )
        fig.show()
else:
    print("  ℹ️  No anomaly data available for visualization")

# Device Performance Clustering
print("\n🎯 Device Performance Clustering...")

# Prepare data for clustering
cluster_features = ['temperature', 'humidity', 'pressure', 'battery_level', 'signal_strength']
device_profiles = sensor_data.groupby('device_id')[cluster_features].mean()

# Perform K-means clustering
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(device_profiles)

# Add cluster labels
device_profiles['cluster'] = clusters

# Visualize clusters using PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(device_profiles[cluster_features])

cluster_viz = pd.DataFrame({
    'PC1': pca_components[:, 0],
    'PC2': pca_components[:, 1],
    'cluster': clusters,
    'device_id': device_profiles.index
})

fig = px.scatter(
    cluster_viz,
    x='PC1',
    y='PC2',
    color='cluster',
    hover_data=['device_id'],
    title='Device Performance Clusters (PCA Visualization)',
    labels={'cluster': 'Cluster Group'}
)
fig.show()

# Cluster analysis
print(f"\n📊 Cluster Analysis Results:")
for i in range(n_clusters):
    cluster_devices = device_profiles[device_profiles['cluster'] == i]
    print(f"\n  Cluster {i}: {len(cluster_devices)} devices")
    print(f"    Avg Temperature: {cluster_devices['temperature'].mean():.1f}°C")
    print(f"    Avg Humidity: {cluster_devices['humidity'].mean():.1f}%")
    print(f"    Avg Battery: {cluster_devices['battery_level'].mean():.1f}%")
    print(f"    Avg Signal: {cluster_devices['signal_strength'].mean():.1f}%")

print(f"\n✅ Advanced analytics complete!")
print(f"📊 Generated {3 + (1 if len(anomaly_data) > 0 else 0)} interactive visualizations")
print(f"🎯 Identified {n_clusters} distinct device performance clusters")